# params-iterable-vs-groups — worked example 1: Flat tensor list normalizes to one group

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `params-iterable-vs-groups`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When an optimizer receives a flat list of tensors, it wraps them into a single parameter group dict so that the rest of the optimizer code always sees a uniform `list[dict]` structure. This dispatch happens at the very start of `__init__` by peeking at the first element. A flat tensor list becomes `[{'params': [...], 'lr': default_lr}]`, preserving the original ordering of tensors.

## Worked solution

**Step 1 — Materialize to list.** The input might be a generator, so we call `list(params)` immediately. This forces evaluation and lets us index into it without fear of exhausting a one-shot iterator.

**Step 2 — Peek at first element.** We check `isinstance(materialized[0], torch.Tensor)`. If True, the caller passed a flat collection of parameters directly (the most common case when building a simple model).

**Step 3 — Wrap in a single group dict.** We return `[{'params': materialized, 'lr': default_lr}]`. Exactly one group, containing all tensors, with the default learning rate filled in. The downstream code never needs to know whether the user passed a list or a dict — it always sees a list of dicts.

**Step 4 — Print the result.** We inspect the output to confirm shape: one group, two keys (`params`, `lr`), and the original tensors present.

In [ ]:
import torch as t

def normalize_flat_params(params, default_lr):
    """Normalize a flat iterable of tensors into a single param-group list."""
    materialized = list(params)
    if not materialized:
        raise ValueError('optimizer got an empty parameter list')
    first = materialized[0]
    if isinstance(first, t.Tensor):
        return [{'params': materialized, 'lr': default_lr}]
    raise TypeError(f'Expected tensors, got {type(first).__name__}')

# Exercise it: three tensors passed as a plain list
t.manual_seed(0)
p1 = t.randn(4, 4)
p2 = t.randn(4)
p3 = t.randn(2, 4)

groups = normalize_flat_params([p1, p2, p3], default_lr=0.01)
print(f'Number of groups: {len(groups)}')              # 1
print(f'Keys: {list(groups[0].keys())}')               # ['params', 'lr']
print(f'Number of tensors: {len(groups[0]["params"])}') # 3
print(f'LR: {groups[0]["lr"]}')                        # 0.01
print(f'Tensors are the original objects: {groups[0]["params"][0] is p1}')  # True